In [ ]:
# IMPORTING LIBRARIES AND TOOLS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib

# ===== Load =====
df = pd.read_csv('/content/drive/MyDrive/Datasets for practice/train.csv')   # apna exact path daalo
# DATA PREPROCESSING
print('Shape of the data set:')
print(df.shape)
print(df.isnull().sum().sum())   # total missing values, 0 expected
print('Total value counts of price_range feature:')
print(df['price_range'].value_counts())

# ===== X, y =====
X = df.drop('price_range', axis=1)
y = df['price_range']

# ===== Split (stratify is important for multiclass) =====
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# ===== Models dictionary =====
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='mlogloss')
}

results = []
trained_pipelines = {}

for name, model in models.items():
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted'),
        'Recall': recall_score(y_test, y_pred, average='weighted'),
        'F1': f1_score(y_test, y_pred, average='weighted')
    })
    trained_pipelines[name] = pipeline

comparison_df = pd.DataFrame(results).round(4)
print(comparison_df)

# ===== Best model select karo (highest F1 ke hisab se) =====
best_model_name = comparison_df.sort_values('F1', ascending=False).iloc[0]['Model']
best_pipeline = trained_pipelines[best_model_name]
print("Best model:", best_model_name)

# ===== Feature importance (agar tree-based model best nikla) =====
if best_model_name in ['Random Forest', 'XGBoost']:
    importances = best_pipeline.named_steps['model'].feature_importances_
    feat_imp = pd.DataFrame({'feature': X.columns, 'importance': importances}).sort_values('importance', ascending=False).head(10)
    plt.figure(figsize=(8,5))
    plt.barh(feat_imp['feature'], feat_imp['importance'])
    plt.gca().invert_yaxis()
    plt.title(f'Top 10 Feature Importances ({best_model_name})')
    plt.show()

# ===== Save best pipeline =====
joblib.dump(best_pipeline, 'mobile_price_pipeline.pkl')

from google.colab import files
files.download('mobile_price_pipeline.pkl')

Shape of the data set:
(2000, 21)
0
Total value counts of price_range feature:
price_range
1    500
2    500
3    500
0    500
Name: count, dtype: int64
                 Model  Accuracy  Precision  Recall      F1
0  Logistic Regression     0.965     0.9650   0.965  0.9650
1        Random Forest     0.880     0.8796   0.880  0.8797
2              XGBoost     0.935     0.9352   0.935  0.9350
Best model: Logistic Regression


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>